# Feature Engineering

**Objective:** Convert raw images and sales data into a structured feature matrix suitable for regression.

**Inputs:**
- `data/processed/product_summary.csv` — per-product aggregates (146 products)
- `data/processed/images_inventory.csv` — 180 image paths with folder labels

**Outputs:**
- `data/processed/image_embeddings.npy` — ResNet50 embeddings, shape (180, 2048)
- `data/processed/training_set.csv` — final feature table with targets

**Sections:**
1. Load processed data
2. Image–product mapping (stratified random assignment)
3. ResNet50 embedding extraction
4. Assemble final training set

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path('..').resolve()
sys.path.append(str(PROJECT_ROOT))

from src.feature_extractor import load_resnet50_extractor, extract_embeddings

PROCESSED = PROJECT_ROOT / 'data' / 'processed'
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

Device: cpu


## 1. Load processed data

In [2]:
products = pd.read_csv(PROCESSED / 'product_summary.csv')
images = pd.read_csv(PROCESSED / 'images_inventory.csv')

print('Products :', len(products))
print('Images   :', len(images))
print('\nProducts head:')
print(products[['code', 'total_qty', 'transactions', 'avg_rate']].head())
print('\nImages head:')
print(images.head())

Products : 146
Images   : 180

Products head:
       code  total_qty  transactions     avg_rate
0    500001       1032            57  1295.017544
1  10016728         24             6   875.000000
2  10019275         56            13  1650.000000
3  10021130         32             7   875.000000
4  10021131         28             6   875.000000

Images head:
   folder                                          filename  \
0       1      WhatsApp Image 2026-04-29 at 1.26.56 PM.jpeg   
1       1  WhatsApp Image 2026-04-29 at 1.26.57 PM (1).jpeg   
2       1      WhatsApp Image 2026-04-29 at 1.26.57 PM.jpeg   
3       1  WhatsApp Image 2026-04-29 at 1.26.58 PM (1).jpeg   
4       1      WhatsApp Image 2026-04-29 at 1.26.58 PM.jpeg   

                                                path  size_kb  
0  E:\POC\project-poc\data\raw\images\1\WhatsApp ...    424.0  
1  E:\POC\project-poc\data\raw\images\1\WhatsApp ...    444.2  
2  E:\POC\project-poc\data\raw\images\1\WhatsApp ...    423.1  
3  E:

## 2. Image–product mapping (stratified random assignment)

**Assumption:** Because filenames do not encode product codes and the four folders do not map to code prefixes, we cannot recover a one-to-one image-to-product mapping from the available data. We therefore generate a *stratified synthetic mapping*:

- Each image is randomly assigned a product code drawn from the full product pool.
- Sampling is performed with replacement (180 images, 146 products).
- The random seed is fixed for reproducibility.
- Folder identity is preserved as a categorical feature and exposes any folder-level demand signal to the model.

**Honest implication:** If image features carry genuine demand signal, the model will learn it and outperform the folder-only baseline. If not, predictions will collapse to folder-level averages — an outcome we will report transparently.

In [3]:
rng = np.random.default_rng(RANDOM_SEED)
assigned_codes = rng.choice(products['code'].values, size=len(images), replace=True)

mapping = images.copy()
mapping['assigned_code'] = assigned_codes
mapping = mapping.merge(
    products[['code', 'total_qty', 'avg_rate', 'transactions']],
    left_on='assigned_code', right_on='code', how='left'
).drop(columns=['code'])

print('Mapping shape:', mapping.shape)
print('Unique codes used :', mapping['assigned_code'].nunique(), 'of', len(products))
print('\nTarget summary (total_qty):')
print(mapping['total_qty'].describe().round(2))
mapping.head()

Mapping shape: (180, 8)
Unique codes used : 100 of 146

Target summary (total_qty):
count    180.00
mean      19.58
std       18.28
min        3.00
25%        8.00
50%       12.00
75%       24.00
max      110.00
Name: total_qty, dtype: float64


,folder,filename,path,size_kb,assigned_code,total_qty,avg_rate,transactions
0,1,WhatsApp Image 2026-04-29 at 1.26.56 PM.jpeg,E:\POC\project-poc\data\raw\images\1\WhatsApp ...,424.0,10027276,12,525.0,3
1,1,WhatsApp Image 2026-04-29 at 1.26.57 PM (1).jpeg,E:\POC\project-poc\data\raw\images\1\WhatsApp ...,444.2,10029305,8,725.0,1
2,1,WhatsApp Image 2026-04-29 at 1.26.57 PM.jpeg,E:\POC\project-poc\data\raw\images\1\WhatsApp ...,423.1,10029229,13,750.0,3
3,1,WhatsApp Image 2026-04-29 at 1.26.58 PM (1).jpeg,E:\POC\project-poc\data\raw\images\1\WhatsApp ...,421.7,10028755,21,550.0,5
4,1,WhatsApp Image 2026-04-29 at 1.26.58 PM.jpeg,E:\POC\project-poc\data\raw\images\1\WhatsApp ...,378.5,10028754,16,550.0,5


In [4]:
print('Folder-wise target distribution after mapping:')
folder_stats = mapping.groupby('folder').agg(
    n_images=('filename', 'size'),
    mean_qty=('total_qty', 'mean'),
    median_qty=('total_qty', 'median'),
    std_qty=('total_qty', 'std'),
    mean_rate=('avg_rate', 'mean'),
).round(2)
folder_stats

Folder-wise target distribution after mapping:


,n_images,mean_qty,median_qty,std_qty,mean_rate
folder,,,,,
1,55,16.31,12.0,14.57,856.10
2,52,19.77,12.0,16.62,869.52
3,36,21.06,18.0,18.30,877.92
4,37,22.73,12.0,24.44,897.70


## 3. ResNet50 embedding extraction

We use ImageNet-pretrained ResNet50 with the final classification layer replaced by identity. Each image is preprocessed (resize 256, centre-crop 224, ImageNet normalisation) and passed through the network. The 2048-dimensional global-average-pooled output forms the visual representation.

In [5]:
model = load_resnet50_extractor(device=device)
print('Model loaded. Parameters:', sum(p.numel() for p in model.parameters()))
print('Output layer:', model.fc)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\praveen/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


0.1%

0.3%

0.4%

0.5%

...

99.7%

99.8%

100.0%

100.0%

Model loaded. Parameters: 23508032
Output layer: Identity()


In [6]:
import time

image_paths = mapping['path'].tolist()
start = time.time()
embeddings = extract_embeddings(image_paths, model=model, device=device, batch_size=16)
elapsed = time.time() - start

print(f'Extracted embeddings : {embeddings.shape}')
print(f'Time taken           : {elapsed:.1f} seconds ({elapsed/len(image_paths):.2f} s/image)')
print(f'Embedding dtype      : {embeddings.dtype}')
print(f'Embedding range      : [{embeddings.min():.3f}, {embeddings.max():.3f}]')

Extracted embeddings : (180, 2048)
Time taken           : 49.8 seconds (0.28 s/image)
Embedding dtype      : float32
Embedding range      : [0.000, 7.160]


In [7]:
np.save(PROCESSED / 'image_embeddings.npy', embeddings)
print('Saved:', PROCESSED / 'image_embeddings.npy')

Saved: E:\POC\project-poc\data\processed\image_embeddings.npy


## 4. Assemble final training set

The final feature matrix combines:
- 2048 image embedding dimensions (from ResNet50)
- 1 numerical feature: average rate of the assigned product
- 4 one-hot encoded folder indicators

Total feature dimensionality: 2053. Target: total quantity sold for the assigned product.

In [8]:
folder_onehot = pd.get_dummies(mapping['folder'].astype(str), prefix='folder').astype(int)
tabular = pd.concat([
    mapping[['assigned_code', 'total_qty', 'avg_rate']].reset_index(drop=True),
    folder_onehot.reset_index(drop=True)
], axis=1)
tabular.head()

,assigned_code,total_qty,avg_rate,folder_1,folder_2,folder_3,folder_4
0,10027276,12,525.0,1,0,0,0
1,10029305,8,725.0,1,0,0,0
2,10029229,13,750.0,1,0,0,0
3,10028755,21,550.0,1,0,0,0
4,10028754,16,550.0,1,0,0,0


In [9]:
embedding_cols = [f'emb_{i}' for i in range(embeddings.shape[1])]
embeddings_df = pd.DataFrame(embeddings, columns=embedding_cols)

training_set = pd.concat([
    mapping[['filename', 'folder', 'path', 'assigned_code']].reset_index(drop=True),
    embeddings_df.reset_index(drop=True),
    mapping[['avg_rate', 'total_qty']].reset_index(drop=True),
    folder_onehot.reset_index(drop=True),
], axis=1)

training_set.to_csv(PROCESSED / 'training_set.csv', index=False)
print('Training set shape :', training_set.shape)
print('Saved              :', PROCESSED / 'training_set.csv')
print('\nColumns (preview):')
print(list(training_set.columns[:5]) + ['...'] + list(training_set.columns[-6:]))

Training set shape : (180, 2058)
Saved              : E:\POC\project-poc\data\processed\training_set.csv

Columns (preview):
['filename', 'folder', 'path', 'assigned_code', 'emb_0', '...', 'avg_rate', 'total_qty', 'folder_1', 'folder_2', 'folder_3', 'folder_4']


In [10]:
feature_cols = embedding_cols + ['avg_rate'] + list(folder_onehot.columns)
target_col = 'total_qty'
print('Feature dimensionality :', len(feature_cols))
print('Target column          :', target_col)
print('Sample rows            :', len(training_set))
print('\nReady for Phase 3 (model training).')

Feature dimensionality : 2053
Target column          : total_qty
Sample rows            : 180

Ready for Phase 3 (model training).
